# From a benzene ring to molecular orbitals
## От бензольного кольца к молекулярным орбиталям

**Educational case storytelling with the Hückel model / Образовательный кейс на основе модели Хюккеля**

This notebook asks a focused question: how can the connectivity of six carbon atoms in benzene produce a structured set of π-molecular orbitals?

Блокнот отвечает на конкретный вопрос: как связность шести атомов углерода в бензоле приводит к формированию системы π-молекулярных орбиталей?

The goal is not to replace quantum-chemical packages. The goal is to build a transparent minimal model that can be inspected line by line.

Цель — не заменить квантово-химические пакеты, а построить прозрачную минимальную модель, которую можно проверить построчно.

## 1. Scientific idea / Научная идея

In the simplest Hückel approximation, every carbon atom contributes one $p_z$ orbital. We keep only nearest-neighbour interactions around the six-membered ring.

В простейшем приближении Хюккеля каждый атом углерода вносит одну $p_z$-орбиталь. Учитываются только взаимодействия ближайших соседей в шестичленном цикле.

The Hamiltonian is written as

$$H = \alpha I + \beta A,$$

where $I$ is the identity matrix and $A$ is the adjacency matrix of the molecular graph. For a qualitative calculation we set $\alpha=0$ and $\beta=-1$.

Гамильтониан записывается как $H = \alpha I + \beta A$, где $A$ — матрица смежности молекулярного графа. Для качественного расчёта используем $\alpha=0$ и $\beta=-1$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n_atoms = 6
alpha = 0.0
beta = -1.0

# Benzene ring: atom i is connected to i-1 and i+1.
adjacency = np.zeros((n_atoms, n_atoms))
for i in range(n_atoms):
    adjacency[i, (i - 1) % n_atoms] = 1
    adjacency[i, (i + 1) % n_atoms] = 1

hamiltonian = alpha * np.eye(n_atoms) + beta * adjacency
hamiltonian

## 2. Solve the eigenvalue problem / Решение задачи на собственные значения

The eigenvalues are orbital energies in units of $|\beta|$. The eigenvectors contain the atomic-orbital coefficients.

Собственные значения соответствуют энергиям орбиталей в единицах $|\beta|$, а собственные векторы содержат коэффициенты атомных орбиталей.

In [ ]:
energies, coefficients = np.linalg.eigh(hamiltonian)

print('Orbital energies / Энергии орбиталей:')
for index, energy in enumerate(energies, start=1):
    print(f'MO {index}: {energy: .3f} |β|')

Benzene has six π electrons. In the ground-state closed-shell picture, the three lowest molecular orbitals are doubly occupied. Therefore, MO 3 is the HOMO and MO 4 is the LUMO.

В бензоле шесть π-электронов. В основном замкнутооболочечном состоянии три нижние молекулярные орбитали заняты по два электрона. Поэтому MO 3 — HOMO, а MO 4 — LUMO.

In [ ]:
n_pi_electrons = 6
n_occupied = n_pi_electrons // 2
homo_index = n_occupied - 1
lumo_index = n_occupied

homo_energy = energies[homo_index]
lumo_energy = energies[lumo_index]
gap = lumo_energy - homo_energy

print(f'HOMO energy: {homo_energy:.3f} |β|')
print(f'LUMO energy: {lumo_energy:.3f} |β|')
print(f'HOMO–LUMO gap: {gap:.3f} |β|')

## 3. Energy-level diagram / Диаграмма энергетических уровней

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for i, energy in enumerate(energies):
    x0 = 0.15 + 0.12 * i
    ax.hlines(energy, x0, x0 + 0.08, linewidth=3)
    occupation = 2 if i < n_occupied else 0
    ax.text(x0 + 0.04, energy + 0.08, f'MO {i + 1}\n{occupation} e⁻', ha='center', fontsize=9)

ax.set_xlim(0.05, 1.0)
ax.set_ylabel('Energy / Энергия, |β|')
ax.set_xticks([])
ax.set_title('Benzene π-orbital energy levels / π-уровни бензола')
ax.grid(axis='y', alpha=0.25)
plt.show()

The pairs of equal energies reflect orbital degeneracy caused by molecular symmetry. This is already a meaningful chemical result obtained from the graph connectivity alone.

Пары одинаковых энергий отражают вырождение орбиталей, связанное с симметрией молекулы. Такой результат получается уже из одной топологии молекулярного графа.

## 4. Inspect HOMO and LUMO coefficients / Коэффициенты HOMO и LUMO

The sign and magnitude of each coefficient describe the phase and contribution of the corresponding carbon $p_z$ orbital.

Знак и модуль коэффициента показывают фазу и вклад $p_z$-орбитали соответствующего атома углерода.

In [ ]:
atom_labels = [f'C{i + 1}' for i in range(n_atoms)]
homo = coefficients[:, homo_index]
lumo = coefficients[:, lumo_index]

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(n_atoms)
width = 0.36
ax.bar(x - width / 2, homo, width, label='HOMO')
ax.bar(x + width / 2, lumo, width, label='LUMO')
ax.axhline(0, linewidth=1)
ax.set_xticks(x, atom_labels)
ax.set_ylabel('MO coefficient / Коэффициент МО')
ax.set_title('Atomic-orbital contributions / Вклады атомных орбиталей')
ax.legend()
plt.show()

## 5. Interpretation / Интерпретация

What the model explains:

- six π molecular orbitals arise from six carbon $p_z$ orbitals;
- orbital energies form a symmetric pattern;
- degeneracy follows from ring symmetry;
- six π electrons fill three bonding orbitals;
- a HOMO–LUMO gap appears naturally.

Что объясняет модель:

- шесть π-молекулярных орбиталей образуются из шести $p_z$-орбиталей;
- энергетические уровни формируют симметричную систему;
- вырождение связано с симметрией цикла;
- шесть π-электронов занимают три связывающие орбитали;
- естественным образом возникает разрыв HOMO–LUMO.

## 6. Model limitations / Ограничения модели

The calculation is qualitative. It does not include electron correlation, σ orbitals, geometry optimisation, solvent, vibronic structure, absolute excitation energies, or transition intensities. Degenerate eigenvectors may also rotate within their shared subspace, so individual coefficient patterns are not unique.

Расчёт носит качественный характер. Он не учитывает электронную корреляцию, σ-орбитали, оптимизацию геометрии, растворитель, колебательную структуру, абсолютные энергии возбуждения и интенсивности переходов. Внутри вырожденного подпространства собственные векторы могут поворачиваться, поэтому отдельный набор коэффициентов не является единственно возможным.

A production scientific workflow would continue with a defined geometry, basis set, electronic-structure method, convergence controls, provenance metadata, and comparison with experimental data.

Полноценный научный workflow должен включать заданную геометрию, базис, метод электронной структуры, контроль сходимости, метаданные происхождения расчёта и сравнение с экспериментом.

## 7. Questions for further work / Вопросы для продолжения

1. What changes if one carbon atom is replaced by nitrogen?
2. How does bond alternation modify the energy gap?
3. Can the model be generalized to naphthalene or another molecular graph?
4. How do Hückel predictions compare with a PySCF calculation?

1. Что изменится при замене одного атома углерода на азот?
2. Как чередование связей повлияет на энергетический разрыв?
3. Как распространить модель на нафталин или другой молекулярный граф?
4. Как результаты модели Хюккеля соотносятся с расчётом в PySCF?